In [1]:
import data_loading
import data_processing
import os
import helper
#CONSTANTS
data_code = 'phrasebank_1020' # this should be updated later
broken_sentence_ids = [1664]
helper.setup_directory(data_code)

<project-root>/directory_phrasebank_1020


# 1 - PREPARING DATASET

The Financial Phrasebank dataset contains 4,840 sentences in total (28.1% Positive, 59.4% Neutral, and 12.5% Negative). Using the entire dataset without Cloud server access is computationally expensive because of the computational complexity of adversarial attack algorithms, and the classes are unbalanced.

To address these issues, a random sample was created with a sample size of 1,020 sentences (340 samples from each class).

In [2]:
#DATA PREPARATION
phrasebank = data_loading.load_financial_phrasebank() # loads data
phrasebank_with_id = data_processing.assign_sentence_id(phrasebank) # adds Sentence_ID
phrasebank_clean = data_processing.remove_broken_sentences(phrasebank_with_id,broken_sentence_ids) #remove broken sentences from dataset
phrasebank_1020 = data_processing.downsample_to_equal_classes(phrasebank_clean,340)  # downsamples data into positive,neutral,negative classes : #340 x 3 , 1020 samples
helper.save_dataset_to_disk(data_code,dataset=phrasebank_1020)

phrasebank_1020 already exists in the directory. No action taken.


# 2 - EXECUTING ATTACKS FROM OPENATTACK FRAMEWORKS

Seven different attack algorithms were chosen and categorized into first, second, and third priority lists.

Four different sentiment models were selected for comparison to answer various research questions.

Each of the eight attack algorithms was executed for each of the four sentiment models using the sample from the Financial Phrasebank dataset.

In [3]:
import openattack_iteration
import nltk
import warnings
nltk.download('wordnet')
warnings.filterwarnings("ignore", message="The multilingual functions are not available with this Wordnet version")
import ssl
import certifi

# Create a default SSL context using certifi
context = ssl.create_default_context(cafile=certifi.where())

[nltk_data] Downloading package wordnet to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
#CONSTANTS
models = helper.bring_4models_list() # DistilRoBERTa-Fin , ROBERTa-base, FinBERT-Araci, FinBERT-Huang
attacks_first_prio = ['SCPN','TF','UAT'] 
attacks_second_prio = ['TBG','DWB'] 
attacks_third_prio = ['PWWS','PSO'] 
ca_bundle_path = 'cert.pem'
sub_dir = 'attack_results_raw' # keep attack results in raw format here
dir_attack_results_raw = os.path.join(os.getcwd(), sub_dir)
if not os.path.exists(dir_attack_results_raw):
    os.makedirs(dir_attack_results_raw)

In [5]:
openattack_iteration.run_attacks(attacks_first_prio, models, data_code, dir_attack_results_raw)

A results file for SCPN against RB already exists in attack results directory. Skipping this attack.
A results file for SCPN against DR already exists in attack results directory. Skipping this attack.
A results file for SCPN against FA already exists in attack results directory. Skipping this attack.
A results file for SCPN against FH already exists in attack results directory. Skipping this attack.
A results file for TF against RB already exists in attack results directory. Skipping this attack.
A results file for TF against DR already exists in attack results directory. Skipping this attack.
A results file for TF against FA already exists in attack results directory. Skipping this attack.
A results file for TF against FH already exists in attack results directory. Skipping this attack.
A results file for UAT against RB already exists in attack results directory. Skipping this attack.
A results file for UAT against DR already exists in attack results directory. Skipping this attack.


In [6]:
openattack_iteration.run_attacks(attacks_second_prio, models, data_code, dir_attack_results_raw)

A results file for TBG against RB already exists in attack results directory. Skipping this attack.
A results file for TBG against DR already exists in attack results directory. Skipping this attack.
A results file for TBG against FA already exists in attack results directory. Skipping this attack.
A results file for TBG against FH already exists in attack results directory. Skipping this attack.
A results file for DWB against RB already exists in attack results directory. Skipping this attack.
A results file for DWB against DR already exists in attack results directory. Skipping this attack.
A results file for DWB against FA already exists in attack results directory. Skipping this attack.
A results file for DWB against FH already exists in attack results directory. Skipping this attack.


In [7]:
openattack_iteration.run_attacks(attacks_third_prio, models, data_code, dir_attack_results_raw)

A results file for PWWS against RB already exists in attack results directory. Skipping this attack.
A results file for PWWS against DR already exists in attack results directory. Skipping this attack.
A results file for PWWS against FA already exists in attack results directory. Skipping this attack.
A results file for PWWS against FH already exists in attack results directory. Skipping this attack.
A results file for PSO against RB already exists in attack results directory. Skipping this attack.
A results file for PSO against DR already exists in attack results directory. Skipping this attack.
A results file for PSO against FA already exists in attack results directory. Skipping this attack.
A results file for PSO against FH already exists in attack results directory. Skipping this attack.


# 3 - MERGING ATTACK RESULTS WITH ORIGINAL DATASET

After generating attack results from OpenAttack framework, we can merge this results with original dataset for further data processing in next steps.

In [8]:
from datasets import load_from_disk
import result_processing
dir_path = os.getcwd()
original_dataset = load_from_disk(data_code)

In [9]:
result_processing.create_results_dataset_for_models(models,original_dataset,data_code,dir_path)

List of attacks found and merged with original dataset:
	TBG_vs_RB_phrasebank_1020
	UAT_vs_RB_phrasebank_1020
	DWB_vs_RB_phrasebank_1020
	TF_vs_RB_phrasebank_1020
	SCPN_vs_RB_phrasebank_1020
	PWWS_vs_RB_phrasebank_1020
	PSO_vs_RB_phrasebank_1020
Saving RB dataset with attack results


Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

List of attacks found and merged with original dataset:
	PSO_vs_DR_phrasebank_1020
	PWWS_vs_DR_phrasebank_1020
	TBG_vs_DR_phrasebank_1020
	DWB_vs_DR_phrasebank_1020
	TF_vs_DR_phrasebank_1020
	UAT_vs_DR_phrasebank_1020
	SCPN_vs_DR_phrasebank_1020
Saving DR dataset with attack results


Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

List of attacks found and merged with original dataset:
	DWB_vs_FA_phrasebank_1020
	PWWS_vs_FA_phrasebank_1020
	TBG_vs_FA_phrasebank_1020
	SCPN_vs_FA_phrasebank_1020
	PSO_vs_FA_phrasebank_1020
	TF_vs_FA_phrasebank_1020
	UAT_vs_FA_phrasebank_1020
Saving FA dataset with attack results


Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

List of attacks found and merged with original dataset:
	TBG_vs_FH_phrasebank_1020
	PSO_vs_FH_phrasebank_1020
	TF_vs_FH_phrasebank_1020
	DWB_vs_FH_phrasebank_1020
	UAT_vs_FH_phrasebank_1020
	SCPN_vs_FH_phrasebank_1020
	PWWS_vs_FH_phrasebank_1020
Saving FH dataset with attack results


Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

# 4 - FILLING NONE ADVERSARIAL SAMPLES

Failed attacks have None values in the sentences, which creates problems for the re-classification of adversarial samples. When an attack fails, it doesn't generate a new sentence, meaning the original sentence could not be changed. To avoid this issue, I will fill the None values with the original sentence (Sentence_Original).


In [10]:
# Define the directory to save the results
dir_datasets_filled_na = os.path.join(dir_path, 'datasets_filled_na')

# Ensure the directory exists
if not os.path.exists(dir_datasets_filled_na):
    os.makedirs(dir_datasets_filled_na)

for model_code in models:
    dataset = result_processing.bring_dataset_with_attack_results(model_code, data_code,dir_path)
    adv_sample_columns = result_processing.filter_adv_sample_columns(dataset)
    dataset_filled_na = result_processing.fill_none_for_failed_attacks(dataset, adv_sample_columns)
    dataset_filled_na.save_to_disk(os.path.join(dir_datasets_filled_na, f'{model_code}_{data_code}_with_results_filled_na'))

print(f"Results saved in {dir_datasets_filled_na}")

Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

Results saved in <project-root>/directory_phrasebank_1020/datasets_filled_na


# 5 - CLASSIFY ALL SENTENCES

After filling all NA sentences for failed attacks, we can re-classify the adversarial samples generated by attack algorithms with each of 4 sentiment models.

In [11]:
import datasets
import logging 
logging.getLogger("datasets").setLevel(logging.ERROR)
import sentiment_classifiers

In [12]:
attack_results_list = os.listdir(dir_datasets_filled_na)

In [13]:
filled_na_dir = os.path.join(dir_path, 'datasets_filled_na')
complete_sentiment_dir = os.path.join(dir_path,'datasets_with_complete_sentiment')
# Ensure the directory exists
if not os.path.exists(complete_sentiment_dir):
    os.makedirs(complete_sentiment_dir)

In [14]:
for attack_result in attack_results_list:
    # Construct the full path to the attack_result within datasets_filled_na
    attack_result_path = os.path.join(filled_na_dir, attack_result)
    
    dataset = datasets.Dataset.load_from_disk(attack_result_path)
    
    # DR, RB, FH, FA etc.
    model_code = helper.extract_model_code(attack_result)
    
    # Bring correct Sentiment Classifier model
    classifier = sentiment_classifiers.initialize_classifier(model_code)
    
    # Bring list of column names, which keeps adversarial examples
    adv_sample_columns = result_processing.filter_adv_sample_columns(dataset)
    print('Classifying adversarial sample columns with', model_code)
    
    # Take all the adversarial example columns, loop over it and add Sentiment columns in dataset.
    dataset = classifier.classify_multiple_columns(dataset, adv_sample_columns)
    print('Completed adversarial examples classification with', model_code, 'and saved sentiments')
    
    # Classify original sentences from dataset
    print('Classifying original sentences')
    dataset = classifier.classify(dataset, 'Sentence')
    print('Completed original sentence classification and saved sentiment\n')
    print('Saving dataset with following columns:', dataset.column_names)
    print(f'Saving {model_code}_{data_code}_complete_sentiment to disk')
    
    complete_path = os.path.join(complete_sentiment_dir, f'{model_code}_{data_code}_complete_sentiment')
    dataset.save_to_disk(complete_path)

Classifying adversarial sample columns with DR
	 1-Classifying column: PSO_result
	 1-Completed column: PSO_result
	 2-Classifying column: PWWS_result
	 2-Completed column: PWWS_result
	 3-Classifying column: TBG_result
	 3-Completed column: TBG_result
	 4-Classifying column: DWB_result
	 4-Completed column: DWB_result
	 5-Classifying column: TF_result
	 5-Completed column: TF_result
	 6-Classifying column: UAT_result
	 6-Completed column: UAT_result
	 7-Classifying column: SCPN_result
	 7-Completed column: SCPN_result
Completed adversarial examples classification with DR and saved sentiments
Classifying original sentences
Completed original sentence classification and saved sentiment

Saving dataset with following columns: ['Sentence_ID', 'Source', 'Sentence', 'Label', 'PSO_success', 'PSO_result', 'PSO_metrics', 'PWWS_success', 'PWWS_result', 'PWWS_metrics', 'TBG_success', 'TBG_result', 'TBG_metrics', 'DWB_success', 'DWB_result', 'DWB_metrics', 'TF_success', 'TF_result', 'TF_metrics',

Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

Classifying adversarial sample columns with FA
	 1-Classifying column: DWB_result
	 1-Completed column: DWB_result
	 2-Classifying column: PWWS_result
	 2-Completed column: PWWS_result
	 3-Classifying column: TBG_result
	 3-Completed column: TBG_result
	 4-Classifying column: SCPN_result
	 4-Completed column: SCPN_result
	 5-Classifying column: PSO_result
	 5-Completed column: PSO_result
	 6-Classifying column: TF_result
	 6-Completed column: TF_result
	 7-Classifying column: UAT_result
	 7-Completed column: UAT_result
Completed adversarial examples classification with FA and saved sentiments
Classifying original sentences
Completed original sentence classification and saved sentiment

Saving dataset with following columns: ['Sentence_ID', 'Source', 'Sentence', 'Label', 'DWB_success', 'DWB_result', 'DWB_metrics', 'PWWS_success', 'PWWS_result', 'PWWS_metrics', 'TBG_success', 'TBG_result', 'TBG_metrics', 'SCPN_success', 'SCPN_result', 'SCPN_metrics', 'PSO_success', 'PSO_result', 'PSO_met

Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

Classifying adversarial sample columns with RB
	 1-Classifying column: TBG_result
	 1-Completed column: TBG_result
	 2-Classifying column: UAT_result
	 2-Completed column: UAT_result
	 3-Classifying column: DWB_result
	 3-Completed column: DWB_result
	 4-Classifying column: TF_result
	 4-Completed column: TF_result
	 5-Classifying column: SCPN_result
	 5-Completed column: SCPN_result
	 6-Classifying column: PWWS_result
	 6-Completed column: PWWS_result
	 7-Classifying column: PSO_result
	 7-Completed column: PSO_result
Completed adversarial examples classification with RB and saved sentiments
Classifying original sentences
Completed original sentence classification and saved sentiment

Saving dataset with following columns: ['Sentence_ID', 'Source', 'Sentence', 'Label', 'TBG_success', 'TBG_result', 'TBG_metrics', 'UAT_success', 'UAT_result', 'UAT_metrics', 'DWB_success', 'DWB_result', 'DWB_metrics', 'TF_success', 'TF_result', 'TF_metrics', 'SCPN_success', 'SCPN_result', 'SCPN_metrics',

Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

Classifying adversarial sample columns with FH
	 1-Classifying column: TBG_result
	 1-Completed column: TBG_result
	 2-Classifying column: PSO_result
	 2-Completed column: PSO_result
	 3-Classifying column: TF_result
	 3-Completed column: TF_result
	 4-Classifying column: DWB_result
	 4-Completed column: DWB_result
	 5-Classifying column: UAT_result
	 5-Completed column: UAT_result
	 6-Classifying column: SCPN_result
	 6-Completed column: SCPN_result
	 7-Classifying column: PWWS_result
	 7-Completed column: PWWS_result
Completed adversarial examples classification with FH and saved sentiments
Classifying original sentences
Completed original sentence classification and saved sentiment

Saving dataset with following columns: ['Sentence_ID', 'Source', 'Sentence', 'Label', 'TBG_success', 'TBG_result', 'TBG_metrics', 'PSO_success', 'PSO_result', 'PSO_metrics', 'TF_success', 'TF_result', 'TF_metrics', 'DWB_success', 'DWB_result', 'DWB_metrics', 'UAT_success', 'UAT_result', 'UAT_metrics', 'S

Saving the dataset (0/1 shards):   0%|          | 0/1020 [00:00<?, ? examples/s]

# 6 - CALCULATING QUANTITATIVE METRICS

Accuracy, precision, recall and F-1 scores are standard metrics to assess the prediction capabilities of sentiment models. 
Attack success rate is metric generated by OpenAttack framework. It is the ratio of successful attacks on different financial sentiment models generated by different adversarial attack algorithms.

Below, these 5 metrics are generated for 4 different sentiment models and 7 different attack algorithms. These can be used to analyze attack performances of different algorithms and robustness of different sentiment models against these attacks.

In [15]:
import evaluation_quant
from IPython.display import display

In [16]:
after_code = 'datasets_complete_sentiment'
search_code = 'complete_sentiment'
search_string = f'{data_code}_{search_code}'

# Define the directory to save the results
dir_datasets_complete_sentiment = os.path.join(dir_path,after_code)

# Ensure the directory exists
if not os.path.exists(dir_datasets_complete_sentiment):
    os.makedirs(dir_datasets_complete_sentiment)

os.chdir(dir_datasets_complete_sentiment) # go into directory

In [17]:
dataset_paths = helper.bring_datasets_list(search_string,dir_datasets_complete_sentiment)
models_list = helper.extract_model_names_from_paths(dataset_paths)

quality_excel_results
FH_phrasebank_1020_complete_sentiment
RB_phrasebank_1020_complete_sentiment
excel_results
directory_phrasebank_1020
DR_phrasebank_1020_complete_sentiment
FA_phrasebank_1020_complete_sentiment
attack_results_raw
phrasebank_1020


In [18]:
# Set the path to the results folder in the current directory
results_folder = 'excel_results'

# Check if the folder exists, and if not, create it
if not os.path.exists(results_folder):
    os.mkdir(results_folder)

for model_code, dataset_path in zip(models_list, dataset_paths):
    dataset = load_from_disk(dataset_path)
    dataset = helper.standardize_col_names(dataset, model_code)  # Standardizing column names
    metrics_model = evaluation_quant.calculate_metrics_for_model(dataset, model_code)  # Calculating initial metrics
    metrics_after_attacks = evaluation_quant.calculate_metrics_after_attacks(dataset, model_code)  # Metrics after attacks
    metrics_changes = evaluation_quant.calculate_metric_changes(dataset, model_code)  # Changes in metrics
    
    '''
    print(metrics_model)
    print(metrics_after_attacks)
    print(metrics_changes)
    '''
    
    # Display each DataFrame in a formatted and readable manner
    display(metrics_model)
    display(metrics_after_attacks)
    display(metrics_changes)
    print("\n" + "-"*80)  # Adds a separator for clarity between each model's output
    
 
    file_path_model = os.path.join(results_folder, f'{model_code}_metrics_model.xlsx')
    file_path_attacks = os.path.join(results_folder, f'{model_code}_metrics_after_attacks.xlsx')
    file_path_changes = os.path.join(results_folder, f'{model_code}_metrics_changes.xlsx')
    metrics_model.to_excel(file_path_model, index=True)
    metrics_after_attacks.to_excel(file_path_attacks, index=True)
    metrics_changes.to_excel(file_path_changes, index=True)

,Accuracy,Precision,Recall,F1-Score
FH,0.724,0.783,0.724,0.726


,Accuracy,Precision,Recall,F1-Score,Attack Success Rate
Attack vs Model,,,,,
TF vs FH,0.340,0.371,0.340,0.324,0.718
PWWS vs FH,0.365,0.392,0.365,0.345,0.688
TBG vs FH,0.373,0.412,0.373,0.324,0.647
SCPN vs FH,0.381,0.498,0.381,0.288,0.493
DWB vs FH,0.392,0.530,0.392,0.315,0.504
PSO vs FH,0.401,0.447,0.401,0.370,0.576
UAT vs FH,0.723,0.777,0.723,0.725,0.025


,Accuracy (Change),Precision (Change),Recall (Change),F1-Score (Change)
Attack vs Model,,,,
TF vs FH,-0.384,-0.412,-0.384,-0.402
PWWS vs FH,-0.359,-0.391,-0.359,-0.381
TBG vs FH,-0.351,-0.371,-0.351,-0.402
SCPN vs FH,-0.343,-0.285,-0.343,-0.438
DWB vs FH,-0.332,-0.253,-0.332,-0.411
PSO vs FH,-0.323,-0.336,-0.323,-0.356
UAT vs FH,-0.001,-0.006,-0.001,-0.001



--------------------------------------------------------------------------------


,Accuracy,Precision,Recall,F1-Score
RB,0.543,0.719,0.543,0.534


...


--------------------------------------------------------------------------------


,Accuracy,Precision,Recall,F1-Score
FA,0.909,0.909,0.909,0.909


,Accuracy,Precision,Recall,F1-Score,Attack Success Rate
Attack vs Model,,,,,
TBG vs FA,0.326,0.321,0.326,0.308,0.745
TF vs FA,0.331,0.333,0.331,0.321,0.738
SCPN vs FA,0.339,0.350,0.339,0.247,0.703
PWWS vs FA,0.356,0.355,0.356,0.346,0.713
PSO vs FA,0.442,0.439,0.442,0.432,0.606
DWB vs FA,0.522,0.577,0.522,0.489,0.518
UAT vs FA,0.904,0.904,0.904,0.904,0.032


,Accuracy (Change),Precision (Change),Recall (Change),F1-Score (Change)
Attack vs Model,,,,
TBG vs FA,-0.583,-0.588,-0.583,-0.601
TF vs FA,-0.578,-0.576,-0.578,-0.588
SCPN vs FA,-0.570,-0.559,-0.570,-0.662
PWWS vs FA,-0.553,-0.554,-0.553,-0.563
PSO vs FA,-0.467,-0.470,-0.467,-0.477
DWB vs FA,-0.387,-0.332,-0.387,-0.420
UAT vs FA,-0.005,-0.005,-0.005,-0.005



--------------------------------------------------------------------------------


# 7 - CALCULATING MINILM SCORE

MiniLM is sentence transformers model and accessible via: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2.
 
This model is used to calculate similarity scores between original sentences and adversarial samples in the section below.



In [19]:
import evaluation_quality
after_code = 'datasets_complete_sentiment'
search_code = 'complete_sentiment'
search_string = f'{data_code}_{search_code}'
dir_path_datasets_final = os.path.join(dir_path,'datasets_final')


In [20]:
dataset_paths

['<project-root>/directory_phrasebank_1020/datasets_complete_sentiment/FH_phrasebank_1020_complete_sentiment',
 '<project-root>/directory_phrasebank_1020/datasets_complete_sentiment/RB_phrasebank_1020_complete_sentiment',
 '<project-root>/directory_phrasebank_1020/datasets_complete_sentiment/DR_phrasebank_1020_complete_sentiment',
 '<project-root>/directory_phrasebank_1020/datasets_complete_sentiment/FA_phrasebank_1020_complete_sentiment']

In [21]:
dir_path

'<project-root>/directory_phrasebank_1020'

In [22]:
for path, model_code in zip(dataset_paths, models_list):
    dataset = datasets.load_from_disk(path)
    dataset = helper.standardize_col_names(dataset, model_code)
    file_name = f'{model_code}_{data_code}_final'
    
    # Construct the full path by combining the directory path and file name
    full_file_path = os.path.join(dir_path_datasets_final, file_name)
    
    # Check if the file name exists in the specified directory
    if not os.path.exists(full_file_path):
        # If the dataset does not exist, save the dataset
        dataset_with_minilm_score = evaluation_quality.calculate_MiniLM_Similarity(dataset)
        dataset_with_minilm_score.save_to_disk(full_file_path)
        print(f"Dataset saved to {full_file_path}.")
    else:
        # If the dataset exists, do not save and print a message instead
        print(f"The directory {full_file_path} already exists. Dataset not saved.")


The directory <project-root>/directory_phrasebank_1020/datasets_final/FH_phrasebank_1020_final already exists. Dataset not saved.
The directory <project-root>/directory_phrasebank_1020/datasets_final/RB_phrasebank_1020_final already exists. Dataset not saved.
The directory <project-root>/directory_phrasebank_1020/datasets_final/DR_phrasebank_1020_final already exists. Dataset not saved.
The directory <project-root>/directory_phrasebank_1020/datasets_final/FA_phrasebank_1020_final already exists. Dataset not saved.


# 8 - CALCULATING QUALITY METRICS

In this section, 6 different quality metrics are generated to analyze the quality of adversarial samples generated by 7 different attack algorithms against 4 different sentiment models.

USE Similarity: Semantic similarity score generated by Universal Sentence Encoder
MiniLM Similarity: Semantic similarity score generated by MiniLM Sentence transformer model
METEOR Score:  The METEOR score (Metric for Evaluation of Translation with Explicit ORdering) is an evaluation metric used to assess the quality of machine-generated text, especially in the context of machine translation, summarization, and other natural language processing tasks. It was developed as an alternative to BLEU (Bilingual Evaluation Understudy) to address some of its limitations.
Fluency: Fluency score generated by GPT-2 perplexity, the lower the better. It is generated by OpenAttack framework.
Grammatical Errors: Average number of grammatical errors, generated by OpenAttack framework.
Word Modification Rate: Average number of word modifications in adversarial samples, generated by OpenAttack framework.


In [23]:
after_code = 'final'
search_string = f'{data_code}_{after_code}'
dataset_paths = helper.bring_datasets_list(search_string,dir_path_datasets_final)
models_list = helper.extract_model_names_from_paths(dataset_paths)

FH_phrasebank_1020_final
DR_phrasebank_1020_final
FA_phrasebank_1020_final
RB_phrasebank_1020_final


In [24]:
output_dir = "quality_excel_results"
os.makedirs(output_dir, exist_ok=True)

for model_code, dataset_path in zip(models_list, dataset_paths):
    dataset = load_from_disk(dataset_path)  # Assuming this function loads the dataset correctly
    dataset = helper.standardize_col_names(dataset, model_code)  # Standardizing column names
    quality_metrics = evaluation_quality.calculate_quality_metrics(dataset,model_code)
    display(quality_metrics)

    print("\n" + "-"*80)  # Adds a separator for clarity between each model's output
    
    # Save the quality metrics to an Excel file
    file_path = os.path.join(output_dir, f"{model_code}_quality_metrics.xlsx")
    quality_metrics.to_excel(file_path, index=True)


,USE Similarity,MiniLM Similarity,METEOR Score,Fluency (GPT-2 perplexity),Grammatical Errors,Word Modification Rate
Attack vs Model (Adv. Samples),,,,,,
UAT vs FH,0.954,0.982,0.945,554,5.2,1.16
PSO vs FH,0.848,0.879,0.861,697,6.6,0.24
PWWS vs FH,0.844,0.869,0.857,648,6.8,0.26
TBG vs FH,0.814,0.815,0.817,715,6.4,0.25
TF vs FH,0.814,0.841,0.837,732,6.8,0.30
DWB vs FH,0.761,0.728,0.709,635,8.6,0.20
SCPN vs FH,0.588,0.666,0.468,537,3.8,1.53



--------------------------------------------------------------------------------


,USE Similarity,MiniLM Similarity,METEOR Score,Fluency (GPT-2 perplexity),Grammatical Errors,Word Modification Rate
Attack vs Model (Adv. Samples),,,,,,
UAT vs DR,0.942,0.984,0.942,525,4.9,1.16
PWWS vs DR,0.865,0.886,0.874,588,6.9,0.25
PSO vs DR,0.855,0.875,0.865,631,6.9,0.25
TF vs DR,0.823,0.857,0.843,692,7.0,0.30
TBG vs DR,0.806,0.806,0.805,784,6.7,0.28
DWB vs DR,0.753,0.736,0.697,739,8.8,0.21
SCPN vs DR,0.581,0.663,0.454,538,4.0,1.61



--------------------------------------------------------------------------------


,USE Similarity,MiniLM Similarity,METEOR Score,Fluency (GPT-2 perplexity),Grammatical Errors,Word Modification Rate
Attack vs Model (Adv. Samples),,,,,,
UAT vs FA,0.938,0.984,0.935,384,4.9,1.16
PSO vs FA,0.851,0.867,0.866,699,6.5,0.25
PWWS vs FA,0.851,0.863,0.865,678,6.7,0.27
TF vs FA,0.820,0.844,0.845,764,6.8,0.29
TBG vs FA,0.807,0.796,0.805,973,6.5,0.30
DWB vs FA,0.760,0.731,0.706,733,8.4,0.21
SCPN vs FA,0.587,0.664,0.464,516,4.0,1.54



--------------------------------------------------------------------------------


,USE Similarity,MiniLM Similarity,METEOR Score,Fluency (GPT-2 perplexity),Grammatical Errors,Word Modification Rate
Attack vs Model (Adv. Samples),,,,,,
UAT vs RB,0.908,0.982,0.903,362,5.2,1.15
PWWS vs RB,0.864,0.882,0.876,839,6.5,0.24
PSO vs RB,0.861,0.877,0.879,755,6.1,0.24
TF vs RB,0.816,0.844,0.841,877,6.5,0.29
TBG vs RB,0.804,0.814,0.808,884,6.4,0.28
DWB vs RB,0.770,0.738,0.711,723,7.9,0.22
SCPN vs RB,0.646,0.724,0.543,453,3.8,1.33



--------------------------------------------------------------------------------


In [25]:
#save datasets again with standard columns
for model_code, dataset_path in zip(models_list, dataset_paths):
    dataset = load_from_disk(dataset_path)  # Assuming this function loads the dataset correctly
    dataset = helper.standardize_col_names(dataset, model_code)  # Standardizing column names
    dataset.to_csv(f'{model_code}_{data_code}_standard_cols.csv')

Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

# 9- COMPARING ADVERSARIAL SAMPLES

In this section, we can analyze and compare adversarial samples for a selected sentiment model. We can analyze the different adversarial samples generated from a same original sentence by different attack algorithms.   

In [26]:
from IPython.display import HTML

# choose a model to analyze adversarial sample sentences
model_code = 'DR' # Distil-Roberta financial sentiment model, check victim_sentiment_models.py for more details

In [27]:
results_one_sample, df_result_one = helper.bring_one_adversarial_sample(model_code, data_code)

for result in results_one_sample:
    print(f"Attack Type: {result['attack_code']}")
    print(f"Sentence ID: {result['sentence_id']}")
    print(f"Original Sentence: {result['original_sentence']}")
    print(f"Adversarial Sample: {result['adversarial_sample']}")
    display(HTML(f"Comparison: {result['highlighted_sample']}"))

Attack Type: PSO
Sentence ID: 2525
Original Sentence: No more waste-burning facilities should be built .
Adversarial Sample: no more waste - burning readiness should be progress .


Attack Type: PWWS
Sentence ID: 2525
Original Sentence: No more waste-burning facilities should be built .
Adversarial Sample: no more waste - burning deftness should be progress .


Attack Type: TF
Sentence ID: 2525
Original Sentence: No more waste-burning facilities should be built .
Adversarial Sample: no more waste - combustion deftness should be progress .
